# Predicting daily PM2.5 in Karaganda, Kazakhstan

Exploratory analysis, model building and evaluation for the combined case study and research
article. Every figure and table in the written deliverables is reproduced here.

**Target**: daily mean PM2.5 (CAMS reanalysis, ug/m3).
**Predictors**: 21 meteorological and calendar variables from ERA5. No pollutant is used as a
predictor, so the coefficient of determination reads directly as the share of daily variance
explained by weather and season.

The notebook calls the same functions as the command-line pipeline, so there is one
implementation of each step and no risk of the notebook and the report disagreeing.

## 1. Setup

`src/` holds the pipeline. `data/raw/` is regenerated by `fetch_open_meteo.py` and the three
bulletin parsers; that step needs network access and is not repeated here.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
print('project root:', ROOT)

## 2. Build the modelling table

`build_features` joins the ERA5 weather record with the CAMS composition record and adds the
derived predictors: the ventilation index (wind speed x mixing depth), heating degree days, and
sine/cosine encodings of wind direction and day of year.

In [ ]:
from build_features import build, feature_columns, TARGET

df = build()
feats = feature_columns(df)
print(f'{len(df)} days, {df.date.min().date()} to {df.date.max().date()}, {len(feats)} predictors')
df[['date', TARGET] + feats[:6]].head()

## 3. Exploratory analysis

### 3.1 Distribution of the target

In [ ]:
print(df[TARGET].describe().round(2).to_string())
print(f'\nskewness {df[TARGET].skew():.2f}  (log-transformed: {np.log(df[TARGET]).skew():.2f})')
print(f'lag-1 autocorrelation {df[TARGET].autocorr(1):.3f}')

WHO_DAILY = 15.0
print(f'days above the WHO 24-hour guideline of {WHO_DAILY} ug/m3: '
      f'{(df[TARGET] > WHO_DAILY).sum()} of {len(df)} '
      f'({100 * (df[TARGET] > WHO_DAILY).mean():.1f}%)')

The target is strongly right-skewed and strongly autocorrelated. Both facts matter later:
the skew because it penalises squared-error linear models, the autocorrelation because it makes a
shuffled cross-validation split optimistic.

### 3.2 Seasonality

In [ ]:
monthly = df.groupby(df.date.dt.month)[TARGET].agg(['mean', 'median', 'max', 'count']).round(2)
monthly.index.name = 'month'
monthly

The reanalysis peaks in March rather than January. This is the 40 km grid cell picking up the
regional spring dust signal, while diluting the localised winter heating plume that the ground
posts sit inside. Section 5.4 of the report develops the consequence.

### 3.3 Correlation with meteorology, split by season

In [ ]:
for label, sub in [('all year', df),
                   ('heating season', df[df.is_heating_season == 1]),
                   ('warm season', df[df.is_heating_season == 0])]:
    r = sub[feats + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
    print(f'{label} (n={len(sub)}): ' + '  '.join(f'{k} {v:+.2f}' for k, v in r.head(4).items()))

The dispersion variables dominate, and every one of them roughly doubles in strength during the
heating season. That is the inversion mechanism showing up in the correlations before any model is
fitted.

### 3.4 How much of the daily variance is the season?

STL with `period=365` is the obvious tool and is the wrong one here: on a four-year record its
seasonal sub-series holds only four observations per day of year, so its smoother cannot
distinguish the annual cycle from synoptic noise and reports most of the weather as "seasonal"
(it gives a seasonal share of 53 % on this data). A smoothed day-of-year climatology does the
separation the question actually needs.

In [ ]:
s = df.set_index('date')[TARGET].asfreq('D').interpolate()
doy = s.index.dayofyear

clim = s.groupby(doy).mean().reindex(range(1, 367)).interpolate()
wrapped = pd.concat([clim.iloc[-31:], clim, clim.iloc[:31]])          # circular, so Dec meets Jan
clim_s = wrapped.rolling(31, center=True, min_periods=1).mean().iloc[31:-31]

seasonal = pd.Series(doy.map(clim_s).values, index=s.index)
trend = (s - seasonal).rolling(365, center=True, min_periods=120).mean()
resid = s - seasonal - trend

comp = pd.DataFrame({'seasonal': seasonal - seasonal.mean(),
                     'trend': trend - trend.mean(), 'residual': resid}).dropna()
share = (comp.var() / s.loc[comp.index].var() * 100).round(1)
print(share.rename('% of daily variance').to_string())
print()
r2_clim = 1 - ((s - seasonal) ** 2).sum() / ((s - s.mean()) ** 2).sum()
print(f'climatology alone as a predictor: R2 = {r2_clim:.3f}')
print(f'peak {clim_s.max():.1f} ug/m3 on day {clim_s.idxmax()}, '
      f'trough {clim_s.min():.1f} on day {clim_s.idxmin()}')

Only 6 per cent of the daily variance is the repeatable annual cycle; 89 per cent is synoptic
weather within the season. Knowing the date predicts a given day with an R2 of 0.08. That single
number frames the whole modelling exercise: almost all of the usable signal is in the weather,
which is why a meteorology-driven model reaches 0.59 while the calendar alone reaches 0.08.
The trend component is negligible: there is no detectable improvement or deterioration over the
four years of record.

## 4. Model benchmark

Eleven algorithms are compared under an identical pipeline. Median imputation applies to every
model; standardisation applies only to the scale-sensitive linear and instance-based ones. Both
steps sit inside the pipeline, so they are refitted on each training fold and never see the
held-out fold. Regularisation strength and neighbourhood size are chosen by an inner
cross-validation on the training fold alone.

In [ ]:
from sklearn.model_selection import KFold, TimeSeriesSplit
from run_models import models, evaluate, lagged_frame, SEED, K

X, y = df[feats], df[TARGET]
kfold = KFold(n_splits=K, shuffle=True, random_state=SEED)

print(f'{len(models())} estimators, including a mean baseline:')
for name in models():
    print(' -', name)

### 4.1 Experiment A: the prescribed shuffled ten-fold protocol

This is Table 1 of the assignment. It takes a couple of minutes.

In [ ]:
summary_a, folds_a = evaluate(X, y, kfold)
summary_a[['Algorithm', 'Number of features', 'k-fold validation',
           'RMSE', 'RMSE_sd', 'MAE', 'R2', 'R2_sd']].round(3)

Three results stand out. The tree ensembles reach an R2 of 0.53 to 0.59 while the regularised
linear models cluster at 0.32, which is the quantitative statement that the relationship is
non-linear. The differences among the leading ensembles are smaller than the fold-to-fold
variation, so the top three are not meaningfully separated from each other, only from everything
below. Classic adaptive boosting is the outlier at 0.22, because reweighting towards the
worst-fit observations concentrates its capacity on a handful of extreme days.

### 4.2 Experiment B: the same benchmark under blocked chronological validation

A shuffled split places days from the same weather episode on both sides of the partition. With a
lag-1 autocorrelation of 0.61 that is a real leak, so the benchmark is repeated with ten expanding
chronological windows.

In [ ]:
blocked = TimeSeriesSplit(n_splits=K)
summary_b, _ = evaluate(X, y, blocked)

merged = (summary_a.set_index('Algorithm')[['RMSE', 'R2']]
          .join(summary_b.set_index('Algorithm')[['RMSE', 'R2']], lsuffix='_shuffled', rsuffix='_blocked'))
merged['R2_drop'] = merged.R2_shuffled - merged.R2_blocked
merged.round(3).sort_values('R2_shuffled', ascending=False)

In [ ]:
# The baseline moves too, so compare skill relative to the matching baseline, not R2 directly.
for label, t in [('A shuffled', summary_a), ('B blocked', summary_b)]:
    t = t.set_index('Algorithm')
    base = t.loc['Baseline (mean)', 'RMSE']
    best = t.drop('Baseline (mean)').RMSE.idxmin()
    print(f'{label}: baseline RMSE {base:.3f} | best {best} at {t.loc[best, "RMSE"]:.3f} '
          f'-> {100 * (1 - t.loc[best, "RMSE"] / base):.1f}% error reduction')

Roughly two fifths of the apparent skill under the prescribed protocol is an artefact of the
shuffled split. The remaining 21 per cent is real forward-looking skill. This is the single most
consequential methodological result of the project: a shuffled ten-fold score on a daily air
quality series is an upper bound, not a generalisation estimate.

### 4.3 Experiment C: next-day forecasting with lagged pollutant history

Adding the previous days' concentrations turns the task into the operationally useful one. It is
evaluated under the same blocked split as experiment B, so the comparison isolates the value of
the lag terms.

In [ ]:
Xc, yc = lagged_frame(df, feats)
summary_c, _ = evaluate(Xc, yc, blocked)
summary_c[['Algorithm', 'Number of features', 'RMSE', 'MAE', 'R2']].round(3).sort_values('RMSE')

## 5. Diagnostics for the best model

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_predict

best = summary_a[summary_a.Algorithm != 'Baseline (mean)'].sort_values('RMSE').Algorithm.iloc[0]
pipe = models()[best]
pred = cross_val_predict(pipe, X, y, cv=kfold)
print(f'{best}: out-of-fold RMSE {np.sqrt(((y - pred) ** 2).mean()):.3f} ug/m3')

pipe.fit(X, y)
imp = permutation_importance(pipe, X, y, n_repeats=20, random_state=SEED,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
(pd.DataFrame({'feature': feats, 'importance': imp.importances_mean})
   .sort_values('importance', ascending=False).head(10).round(3).reset_index(drop=True))

The engineered ventilation index outranks both of its own constituents, which is direct evidence
that the multiplicative interaction of wind speed and mixing depth carries information neither
variable carries alone.

The out-of-fold RMSE printed above pools all predictions before squaring, while Table 1 averages
the ten per-fold RMSE values. The two are not the same quantity and differ by a few hundredths;
Table 1 reports the per-fold mean, which is what the assignment asks for.

In [ ]:
# Residuals against observed concentration: is the model calibrated across the range?
resid = pd.DataFrame({'observed': y, 'predicted': pred, 'residual': y - pred})
bins = pd.cut(resid.observed, [0, 5, 10, 15, 20, 30, 100])
resid.groupby(bins, observed=True).agg(
    n=('residual', 'size'), mean_residual=('residual', 'mean'), mean_observed=('observed', 'mean')
).round(2)

The model is well calibrated in the body of the distribution and systematically underpredicts
above roughly 20 ug/m3. That is the expected behaviour of a squared-error objective on a
heavy-tailed target, and it is the wrong bias for an early-warning system: a deployment should
refit on a quantile objective targeting an upper conditional quantile.

## 6. External validation against measured episodes

Everything above is fitted to a reanalysis product. The 169 high-pollution days recorded by
Kazhydromet are instrument measurements and provide an independent check on whether the learned
mechanism is real.

In [ ]:
raw = pd.read_csv(ROOT / 'data/raw/open_meteo/karaganda_daily.csv', parse_dates=['date'])
heating = raw[(raw.date.dt.year <= 2025) & raw.date.dt.month.isin([1, 2, 3, 10, 11, 12])]

cols = ['temperature_2m_mean', 'wind_speed_10m_mean', 'blh_min_m', 'surface_pressure_mean']
comparison = heating.groupby('vz_day')[cols].median().T
comparison.columns = ['ordinary day', 'high-pollution day']
print(f'{int(heating.vz_day.sum())} episode days vs {int((heating.vz_day == 0).sum())} ordinary heating-season days')
comparison.round(1)

On episode days the median wind speed halves and the median minimum mixing depth falls from 135
to 30 metres. The two variables that separate measured episodes most sharply are the two the model
relies on most, which is the external corroboration an internal cross-validation score cannot
supply.

## 7. Reproducing the full deliverables

The notebook covers the analysis. The complete pipeline, including the figures and the generated
documents, runs from the command line:

```bash
python src/fetch_open_meteo.py     # refresh the reanalysis data (needs network)
python src/build_features.py       # -> data/processed/karaganda_modelling.csv
python src/run_models.py           # -> results/table1*.csv and diagnostics
python src/figures.py              # -> results/figures/*.png
python src/make_documents.py       # -> docs/*.docx and docs/slides.pptx
```

Each script ends in assertion-based self-checks and fails loudly rather than emitting wrong
numbers silently.